<a href="https://colab.research.google.com/github/jackevansadl/chem2002/blob/main/C1/02_RunMD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 2: Running a Molecular Dynamics Simulation ⚛️

Welcome to the second notebook! We've successfully built a static "box of argon" in the last section. Now, it's time to bring it to life. In this notebook, we'll use **Molecular Dynamics (MD)** to simulate the movement of the atoms over time according to the laws of physics.

First, let's set up our environment and build the system.

In [ ]:
!pip install -q ase

In [ ]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
from ase.io import read, write, Trajectory
from ase.build import bulk
from ase import units
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from ase.md import MDLogger

### **📋 Reporting Task 3**

1. Modify the simulation box of below to represent liquid Argon at 100 K and 40 bar with a **fcc** lattice constant of 5.8458 Å. Demonstrate that this results in the correct density according to NIST webbook (webbook.nist.gov) and your pre-practical preparation.
2. Modify the number of repeated units such that the simulation cell lengths more than 20 Å.

In [ ]:
# Build a simulation box of argon atoms in the fcc lattice.
# The fcc lattice constant below is a sensible starting point for liquid argon;
# in Reporting Task 3 you will verify the density it produces against NIST
# (webbook.nist.gov) for 100 K and 40 bar.
argon_bulk = bulk('Ar', 'fcc', a=5.8458, cubic=True)

# Repeat the unit cell so that every cell length is greater than 20 Angstrom.
simulation_box = argon_bulk.repeat((4, 4, 4))
print(f"Created a simulation box with {len(simulation_box)} Argon atoms.")

# Print the simulation box cell dimensions (units of ASE are Angstrom)
print("Cell lengths (Angstrom):", simulation_box.cell.lengths())

# TODO (Reporting Task 3): compute the density of this box and compare to NIST

# Let's visualize our new simulation box
from ase.visualize import view
view(simulation_box, viewer='x3d')

-----

## 2.1 Defining the Physics: The Force Field

How do atoms know how to interact? They follow a set of rules and equations that describe the potential energy when they get close to each other. This "rulebook" is called a **force field**.

For this introductory workshop, we'll use the simple but powerful **Lennard-Jones (LJ) potential**. It models atoms as soft spheres that are attracted at a distance but repel strongly when they get too close.

### Visualizing the Lennard-Jones Potential

Let's plot the LJ potential to understand its features. The potential energy ($E_{LJ}$) between two atoms is a function of the distance ($r$) between them:

$$E_{LJ} = 4\epsilon \left[ \left(\frac{\sigma}{r}\right)^{12} - \left(\frac{\sigma}{r}\right)^{6} \right]$$

The two terms represent the strong short-range **repulsion** and the weaker long-range **attraction** (van der Waals forces), respectively.

In [ ]:
# Plotting the Lennard-Jones Potential
# Parameters for Argon (for visualization purposes)
sigma = 3.4  # Angstrom
epsilon = 0.0103  # eV

# Generate a range of distances
r = np.linspace(3, 10, 100)

# Calculate LJ potential energy
E_lj = 4 * epsilon * ((sigma / r)**12 - (sigma / r)**6)

# Plotting
plt.figure(figsize=(8, 5))
plt.plot(r, E_lj, lw=3)
plt.axhline(0, color='gray', linestyle='--')
plt.title('Lennard-Jones Potential')
plt.xlabel('Distance (r) [Å]')
plt.ylabel('Potential Energy (E) [eV]')
plt.ylim(-0.012, 0.01)
plt.grid(True)
plt.show()

### **📋 Reporting Task 4**

Look at the plot of the Lennard-Jones potential you just generated. For your workbook, report the following:

1.  Estimate the **distance (r)** at which the potential energy is at its **minimum**. What does this distance represent physically?
2.  What happens to the energy at very short distances? What physical interaction does this represent?


In [ ]:
# Answer goes here

-----

Now, we'll apply this potential to our system, this time using the known experimental parameters for Argon to make our simulation accurate.


In [ ]:
# Import the Lennard-Jones calculator from ASE
from ase.calculators.lj import LennardJones

# Parameters for Argon from Nichele et al. (2017)
sigma_argon = 3.405          # Angstrom
epsilon_argon = 0.010321     # eV  (119.8 K * 8.617e-5 eV/K)

# Standard Lennard-Jones cutoff of 2.5*sigma (~8.5 Angstrom): comfortably less
# than half the box length, which keeps the simulation fast and well-behaved.
rc_argon = 2.5 * sigma_argon

# Apply the Lennard-Jones calculator
simulation_box.calc = LennardJones(sigma=sigma_argon,
                                   epsilon=epsilon_argon,
                                   rc=rc_argon)

-----

## 2.2 Setting the Simulation in Motion

 To simulate liquid Argon, we must set the temperature above its melting point (~84 K). We will aim for 100 K. Our simulation will be run in the **NVT ensemble** (constant **N**umber of atoms, **V**olume, and **T**emperature) using a Langevin thermostat.


In [ ]:
# Set the target temperature for liquid Argon
liquid_temp_K = 100.0

# Set up the MD integrator using Langevin dynamics using a 1 fs timestep
dyn = Langevin(simulation_box,
               timestep=2 * units.fs,
               temperature_K=liquid_temp_K,
               friction=0.001/units.fs)


# Create a Trajectory object to write to the file 'md.traj'
traj = Trajectory('md.traj', 'w', simulation_box)
# Attach the 'write' method of the Trajectory object to the simulation
dyn.attach(traj.write, interval=100) # Save every 100 steps

# The MDLogger is set up correctly as it is an object passed to attach
dyn.attach(MDLogger(dyn, simulation_box, 'md.log', header=True,
                  mode='w'), interval=100) # Log every 100 steps


Now we're ready to run! This step takes roughly one to two minutes to complete as it calculates the forces and updates the positions of all atoms for **10,000 steps** (20 ps at a 2 fs timestep).


In [ ]:
print("Starting MD simulation...")
# Run for 10,000 steps, which is 20 ps with a 2 fs timestep.
dyn.run(10000)
print("MD simulation finished!")


-----

## 2.3 Checking for Equilibration

Did our simulation work? A key check is to see if the system reached and maintained the target temperature. This is called **thermal equilibration**.

### Plotting Temperature vs. Time

Let's read the log file (`md.log`) and plot the temperature at each saved step. We expect to see the temperature fluctuate around our target of 100 K.

In [ ]:
# Read the log file
log_data = np.loadtxt('md.log', skiprows=1)

# Extract time (in fs) and temperature (in K)
time = log_data[:, 0]
temperature = log_data[:, 4]

# Plotting
plt.figure(figsize=(8, 5))
plt.plot(time, temperature)
plt.axhline(100, color='r', linestyle='--', label='Target Temperature (100 K)')
plt.title('System Temperature During Simulation')
plt.xlabel('Time [ps]')
plt.ylabel('Temperature [K]')
plt.legend()
plt.grid(True)
plt.show()

### **📋 Reporting Task 5**

Analyze the **Temperature vs. Time** plot you just created. In your workbook, answer the following:

1.  Does the system's temperature reach the target of 100 K?
2.  Approximately how long (in picoseconds) does it take for the temperature to stop making large-scale adjustments and begin fluctuating around a stable average? This point marks the beginning of **equilibration**.
3.  What if you change the `friction` parameter to 0.1? Test this by rerunning shorter simulations.

# Insert your answer here

## Next Steps

Fantastic\! You've run an MD simulation and confirmed that it reached thermal equilibrium. You've also visualized the potential that governs atomic interactions and the resulting motion.

➡️ Now, saving your MD.log file  proceed to the final notebook, **`03-Analysis.ipynb`**, to use the equilibrated part of the simulation to calculate the heat capacity.